# Gradio Demo on Kaggle (with ngrok public URL)

Runs the dual-mode CXR Intelligence system as a Gradio app, exposed via ngrok so anyone can click the URL.

**Setup:**
1. Get a free ngrok token at https://dashboard.ngrok.com/get-started/your-authtoken
2. Add it to Kaggle Secrets as `NGROK_TOKEN`
3. Run the cells below in order — the public URL prints in the last cell

In [ ]:
# 1. Install deps
!pip install -q gradio pyngrok

In [ ]:
# 2. Recovery cell — same paths/env as the eval notebook
import os, sys, pathlib
sys.path.insert(0, '/kaggle/working/cxr-intel/src')
os.chdir('/kaggle/working/cxr-intel')

dst = pathlib.Path('/kaggle/working/cxr-intel/data/raw')
if not dst.exists():
    dst.symlink_to('/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset')

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['NVIDIA_API_KEY'] = secrets.get_secret('NVIDIA_API_KEY')
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['QA_SYNTH_MODEL'] = 'meta/llama-3.1-8b-instruct'
ngrok_token = secrets.get_secret('NGROK_TOKEN')

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 3. Load models (one-time, ~3 min)
from cxr_intel.models.medgemma_runner import MedGemmaRunner
from cxr_intel.retrieval.colpali_index import ColPaliRetriever
from cxr_intel.retrieval.biomedclip_index import BiomedCLIPRetriever

print('Loading MedGemma (INT4)...')
medgemma = MedGemmaRunner(quantization='int4')
medgemma.load()

print('Loading ColPali (patched)...')
colpali = ColPaliRetriever(
    name='colpali_zs',
    checkpoint='/kaggle/working/colpali-v1.3-patched',
    torch_dtype='float16',
)
colpali.load('data/indices/colpali_zs')

print('Loading BiomedCLIP...')
biomedclip = BiomedCLIPRetriever()
biomedclip.load('data/indices/biomedclip')

print(f'\n✓ All loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# 4. Pipeline functions
from cxr_intel.generation.report_pipeline import ReportPipeline
from cxr_intel.generation.qa_pipeline import QAPipeline

RETRIEVERS = {
    'ColPali': colpali,
    'BiomedCLIP': biomedclip,
    'None (Pure MedGemma)': None,
}
CONFIG_NAMES = {
    'ColPali': 'colpali_zs_rag',
    'BiomedCLIP': 'biomedclip_rag',
    'None (Pure MedGemma)': 'medgemma_only',
}

def generate_report(image, retriever_name, top_k):
    retriever = RETRIEVERS[retriever_name]
    config = CONFIG_NAMES[retriever_name]
    pipe = ReportPipeline(config=config, retriever=retriever, medgemma=medgemma, top_k=int(top_k))
    out = pipe.run(image)
    evidence = '\n\n'.join(
        f'[{i+1}] study={h.study_id} (sim={h.score:.3f})\n{h.report_text[:300]}...'
        for i, h in enumerate(out.retrieved)
    ) or 'No retrieval (pure VLM)'
    return out.report_text, evidence, f'{out.latency_s:.1f} s'

def answer_question(image, question, retriever_name, top_k):
    if not question:
        return 'Please enter a question.', '', ''
    retriever = RETRIEVERS[retriever_name]
    config = CONFIG_NAMES[retriever_name]
    pipe = QAPipeline(config=config, retriever=retriever, medgemma=medgemma, top_k=int(top_k))
    out = pipe.run(image, question)
    evidence = '\n\n'.join(
        f'[{i+1}] study={h.study_id} (sim={h.score:.3f})\n{h.report_text[:300]}...'
        for i, h in enumerate(out.retrieved)
    ) or 'No retrieval (pure VLM)'
    return out.answer, evidence, f'{out.latency_s:.1f} s'

In [ ]:
# 5. Gradio UI
import gradio as gr
import os, glob

# Optional: pre-load 3 sample CXRs from the test set for quick demo
import pandas as pd, json
df = pd.read_parquet('data/processed/reports.parquet')
splits = json.load(open('data/processed/splits.json'))
test_df = df[df['study_id'].astype(str).isin(splits['test'])].head(5)
examples_report = [[row['image_path'], 'ColPali', 3] for _, row in test_df.iterrows()]
examples_qa = [[row['image_path'], q, 'ColPali', 3] for _, row in test_df.iterrows() for q in ['Is there cardiomegaly?', 'Describe the principal findings.']][:5]

with gr.Blocks(title='CXR Intelligence — DSAI 413 A2') as demo:
    gr.Markdown('# Multi-Modal Chest X-Ray Intelligence System\nDSAI 413 Assignment 2 — Report Generation + RAG-based QA')
    with gr.Tabs():
        with gr.Tab('Report Generation'):
            with gr.Row():
                with gr.Column():
                    img_r = gr.Image(type='pil', label='Chest X-ray')
                    ret_r = gr.Radio(list(RETRIEVERS.keys()), value='ColPali', label='Retriever')
                    k_r = gr.Slider(0, 5, value=3, step=1, label='Top-K retrievals')
                    btn_r = gr.Button('Generate Report', variant='primary')
                with gr.Column():
                    out_r = gr.Textbox(label='Generated report', lines=10)
                    lat_r = gr.Textbox(label='Latency')
                    with gr.Accordion('Retrieved evidence', open=False):
                        ev_r = gr.Textbox(label='', lines=10)
            btn_r.click(generate_report, [img_r, ret_r, k_r], [out_r, ev_r, lat_r])
            gr.Examples(examples_report, [img_r, ret_r, k_r])

        with gr.Tab('QA Mode'):
            with gr.Row():
                with gr.Column():
                    img_q = gr.Image(type='pil', label='Chest X-ray')
                    q_q = gr.Textbox(label='Question', placeholder='e.g., Is there pleural effusion?')
                    ret_q = gr.Radio(list(RETRIEVERS.keys()), value='ColPali', label='Retriever')
                    k_q = gr.Slider(0, 5, value=3, step=1, label='Top-K retrievals')
                    btn_q = gr.Button('Answer', variant='primary')
                with gr.Column():
                    out_q = gr.Textbox(label='Answer', lines=4)
                    lat_q = gr.Textbox(label='Latency')
                    with gr.Accordion('Retrieved evidence', open=False):
                        ev_q = gr.Textbox(label='', lines=10)
            btn_q.click(answer_question, [img_q, q_q, ret_q, k_q], [out_q, ev_q, lat_q])
            gr.Examples(examples_qa, [img_q, q_q, ret_q, k_q])

        with gr.Tab('Compare all 3 retrievers (Report)'):
            with gr.Row():
                img_c = gr.Image(type='pil', label='Chest X-ray')
                btn_c = gr.Button('Compare', variant='primary')
            with gr.Row():
                with gr.Column():
                    gr.Markdown('### MedGemma only'); out_c1 = gr.Textbox(lines=8); lat_c1 = gr.Textbox(label='latency')
                with gr.Column():
                    gr.Markdown('### BiomedCLIP-RAG'); out_c2 = gr.Textbox(lines=8); lat_c2 = gr.Textbox(label='latency')
                with gr.Column():
                    gr.Markdown('### ColPali-RAG'); out_c3 = gr.Textbox(lines=8); lat_c3 = gr.Textbox(label='latency')
            def compare_all(image):
                r1, _, l1 = generate_report(image, 'None (Pure MedGemma)', 0)
                r2, _, l2 = generate_report(image, 'BiomedCLIP', 3)
                r3, _, l3 = generate_report(image, 'ColPali', 3)
                return r1, l1, r2, l2, r3, l3
            btn_c.click(compare_all, [img_c], [out_c1, lat_c1, out_c2, lat_c2, out_c3, lat_c3])

In [ ]:
# 6. Launch with ngrok public URL
from pyngrok import ngrok
ngrok.set_auth_token(ngrok_token)

PORT = 7860
public_url = ngrok.connect(PORT).public_url
print(f'\n🌐 Public demo URL: {public_url}\n')
print('Paste this URL into report/demo_video_link.md and the README.\n')

demo.launch(server_name='0.0.0.0', server_port=PORT, share=False, quiet=True)